# Image-to-Text (ITT) — vision-language benchmark on Apple Silicon

Runs every image of a prompt set through a vision-language model with **MLX** (`mlx-vlm`) in **float16** and records,
per image: image size, input / output tokens, output words, first token, time to first token, prompt processing rate,
inference rate and inference time, plus system metrics (CPU / GPU / NPU utilisation, power, temperature) sampled by
`macmon`.

Each model is converted once from its original Hugging Face checkpoint into a local float16 MLX copy at
`models/<model>-fp16/`, then loaded from there.

**Outputs**
- `outputs/itt/<family>/<params>/<prompt set>/`: one response file per image and `<model>_simple_metrics.json`
- `metrics_analysis/itt/<family>/<params>/<prompt set>/`: `Model_Level_Metrics/<model>_prompt_metrics.csv`
  and `System_Level_Metrics/<prompt id>_system_metrics.csv`

## 1. Settings and model registry

In [ ]:
import gc
import json
import os
import shutil
import subprocess
import sys
import tempfile
import time
from pathlib import Path

import mlx.core as mx
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'prompts').is_dir())   # project root
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

# family        size      params  model                      original checkpoint (Hugging Face, ungated)  params (B)
REGISTRY = [
    ('Qwen2-VL',   'SMALL',  '2B',   'Qwen2-VL-2B-Instruct',    'Qwen/Qwen2-VL-2B-Instruct',              2.21),
    ('Qwen2-VL',   'LARGE',  '7B',   'Qwen2-VL-7B-Instruct',    'Qwen/Qwen2-VL-7B-Instruct',              8.29),
    ('Qwen2.5-VL', 'SMALL',  '3B',   'Qwen2.5-VL-3B-Instruct',  'Qwen/Qwen2.5-VL-3B-Instruct',            3.75),
    ('Qwen2.5-VL', 'MEDIUM', '7B',   'Qwen2.5-VL-7B-Instruct',  'Qwen/Qwen2.5-VL-7B-Instruct',            8.29),
    ('Qwen2.5-VL', 'LARGE',  '32B',  'Qwen2.5-VL-32B-Instruct', 'Qwen/Qwen2.5-VL-32B-Instruct',          33.45),
    ('Gemma-3',    'SMALL',  '4B',   'gemma-3-4b-it',           'unsloth/gemma-3-4b-it',                  4.30),
    ('Gemma-3',    'MEDIUM', '12B',  'gemma-3-12b-it',          'unsloth/gemma-3-12b-it',                12.19),
    ('Gemma-3',    'LARGE',  '27B',  'gemma-3-27b-it',          'unsloth/gemma-3-27b-it',                27.43),
    ('SmolVLM',    'SMALL',  '256M', 'SmolVLM-256M-Instruct',   'HuggingFaceTB/SmolVLM-256M-Instruct',    0.26),
    ('SmolVLM',    'MEDIUM', '500M', 'SmolVLM-500M-Instruct',   'HuggingFaceTB/SmolVLM-500M-Instruct',    0.51),
    ('SmolVLM',    'LARGE',  '2.2B', 'SmolVLM-Instruct',        'HuggingFaceTB/SmolVLM-Instruct',         2.25),
]
MODELS = pd.DataFrame(REGISTRY, columns=['family', 'size', 'params', 'model', 'source', 'params_b']).set_index('model')
MODELS['path'] = 'models/' + MODELS.index + '-fp16'           # local float16 MLX copy of each model
MODELS['fp16_gb'] = (MODELS.params_b * 2).round(1)             # 2 bytes per parameter
GPU_BUDGET_GB = mx.device_info()['max_recommended_working_set_size'] / 1e9
MODELS['fits'] = MODELS.fp16_gb < 0.9 * GPU_BUDGET_GB          # leave room for activations and macOS

# Pick the model here, or from the terminal with ITT_FAMILY / ITT_SIZE / ITT_PROMPTS / ITT_COMPARE
FAMILY = os.environ.get('ITT_FAMILY', 'Qwen2-VL')   # Qwen2-VL | Qwen2.5-VL | Gemma-3 | SmolVLM
SIZE = os.environ.get('ITT_SIZE', 'LARGE')          # SMALL | MEDIUM | LARGE (only the sizes listed)
MODEL = MODELS.query('family == @FAMILY and size == @SIZE').index[0]
PROMPT_SET = os.environ.get('ITT_PROMPTS', 'itt_prompts_10')   # prompts/images_itt_prompts/<set>.csv
MAX_NEW_TOKENS = min(int(os.environ.get('ITT_MAX_TOKENS', 1024)), 1024)   # hard cap of 1,024 output tokens
COMPARE_FAMILY = os.environ.get('ITT_COMPARE', '1') == '1'       # section 5: also run the family's other models

print(f'Selected: {FAMILY} / {SIZE} -> {MODEL} -> {MODELS.path[MODEL]}')
print(f'GPU memory budget: {GPU_BUDGET_GB:.1f} GB')
MODELS[MODELS.family == FAMILY][['size', 'params', 'source', 'path', 'fp16_gb', 'fits']]

## 2. Load a model

The first time a model is used, its original checkpoint is downloaded and converted to a float16 MLX model in
`models/<model>-fp16/` (one-time step); after that it loads from disk. A model whose float16 weights don't fit in
this Mac's GPU memory stops with a clear error.

In [ ]:
from huggingface_hub import snapshot_download
from mlx_vlm import convert, load, stream_generate
from mlx_vlm.prompt_utils import apply_chat_template

def load_model(name):
    m = MODELS.loc[name]
    if not m.fits:
        raise MemoryError(f"{name} needs {m.fp16_gb} GB in float16, more than this Mac's GPU can use ({GPU_BUDGET_GB:.1f} GB)")
    if not Path(m.path).exists():
        print(f'Converting {m.source} -> {m.path} (float16), one-time step...')
        src = Path(snapshot_download(m.source, token=False, allow_patterns=['*.json', '*.safetensors', '*.py', '*.model',
                                                                            '*.txt', '*.jinja', '*.tiktoken']))
        with tempfile.TemporaryDirectory() as tmp:
            # Convert from a staging folder: weights linked, small files copied as writable files
            # (some cached files are read-only, which breaks the converter's tokenizer re-save)
            for f in src.iterdir():
                if f.suffix == '.safetensors':
                    (Path(tmp) / f.name).symlink_to(f.resolve())
                elif f.is_file():
                    shutil.copyfile(f, Path(tmp) / f.name)
            try:
                convert(hf_path=tmp, mlx_path=m.path, dtype='float16')
            except Exception:
                shutil.rmtree(m.path, ignore_errors=True)   # don't leave a half-converted model behind
                raise
    t0 = time.perf_counter()
    model, processor = load(m.path)
    mx.eval(model.parameters())
    return model, processor, time.perf_counter() - t0

## 3. Prompts

Each row of `prompts/images_itt_prompts/<prompt set>.csv` is one image and the question asked about it. Images are used
as supplied (no resizing).

In [ ]:
from PIL import Image

PROMPTS = pd.read_csv(Path('prompts/images_itt_prompts') / f'{PROMPT_SET}.csv', dtype={'id': str})
PROMPTS['id'] = PROMPTS.id.str.zfill(2)
PROMPTS['image_size'] = ['{}x{}'.format(*Image.open(p).size) for p in PROMPTS.image]
print(f'{len(PROMPTS)} images')
PROMPTS[['id', 'image', 'image_size', 'prompt']]

## 4. Run the selected model

Greedy decoding (temperature 0), so results are reproducible. Each image is wrapped in `metrics_monitor_start()` / `metrics_monitor_stop()` (from
`macmon_metrics/`), which saves that image's system metrics every 100 ms. The summary JSON averages the per-image
metrics and the system metrics over the whole run.

In [ ]:
import macmon_metrics.metrics_monitor as mm
from macmon_metrics.metrics_monitor import metrics_monitor_start, metrics_monitor_stop, setup_output_directory

if mm._tracker is None:   # start macmon now, so every image's capture has samples from its first moment
    mm._tracker = mm.MacMonTracker(binary_path=mm._resolve_macmon_binary())
    mm._tracker.start()
    time.sleep(1.5)

def media(processor, image):
    # (generate kwargs, chat template kwargs, frames) for one image
    return dict(image=[image]), dict(num_images=1), None

def answer(model, processor, image, prompt, max_tokens=MAX_NEW_TOKENS):
    gen_kwargs, template_kwargs, n_frames = media(processor, image)
    formatted = apply_chat_template(processor, model.config, prompt, **template_kwargs)
    text, first, ttft, t0 = '', None, None, time.perf_counter()
    for r in stream_generate(model, processor, formatted, max_tokens=max_tokens, temperature=0.0, **gen_kwargs):
        if ttft is None:
            ttft = time.perf_counter() - t0   # first token arrives once the image and prompt have been processed
        if first is None and r.text:
            first = r.text                    # some models stream an empty piece first
        text += r.text
    return text.strip(), first, ttft, time.perf_counter() - t0, n_frames, r

def run_model(name):
    # Run every image through one model; save responses, per-image metrics and a summary
    m = MODELS.loc[name]
    sub = Path(m.family) / m.params / PROMPT_SET
    out_dir = Path('outputs/itt') / sub
    out_dir.mkdir(parents=True, exist_ok=True)
    sys_dir, model_dir = setup_output_directory(str(Path('metrics_analysis/itt') / sub))

    model, processor, load_s = load_model(name)
    answer(model, processor, PROMPTS.image[0], PROMPTS.prompt[0], max_tokens=8)   # warm-up: compile GPU kernels
    rows, hit_limit, peak_gb = [], 0, 0.0
    for p in PROMPTS.itertuples():
        metrics_monitor_start(idx=p.id, metrics_folder=sys_dir)
        text, first, ttft, total, n_frames, r = answer(model, processor, p.image, p.prompt)
        metrics_monitor_stop()
        hit_limit += r.finish_reason == 'length'
        peak_gb = max(peak_gb, r.peak_memory)
        (out_dir / f'{name}_response_{p.id}.txt').write_text(text + '\n')
        rows.append({
            'prompt_id': p.id, 'model': name, 'family': m.family, 'size': m['size'],
            'image_size (px)': p.image_size,
            'input_tokens': r.prompt_tokens, 'output_tokens': r.generation_tokens, 'output_words': len(text.split()),
            'first_token': json.dumps(first)[1:-1],   # escaped, so newlines stay visible in the CSV
            'time_to_first_token (ms)': round(1000 * ttft, 1),
            'prompt_processing_rate (tokens/s)': round(r.prompt_tps or r.prompt_tokens / ttft, 1),
            'inference_rate (tokens/s)': round(r.generation_tps, 2),
            'inference_time (s)': round(total, 3),
            'image': p.image, 'prompt': p.prompt, 'response': text,
        })
        print(f'  [{p.id}] {r.prompt_tokens} in -> {r.generation_tokens} out, first token {1000 * ttft:.0f} ms, '
              f'{r.generation_tps:.1f} tok/s, {total:.1f} s')
    del model, processor
    gc.collect()
    mx.clear_cache()

    df = pd.DataFrame(rows)
    df.to_csv(Path(model_dir) / f'{name}_prompt_metrics.csv', index=False)
    hw = pd.concat([pd.read_csv(Path(sys_dir) / f'{i}_system_metrics.csv') for i in df.prompt_id])
    summary = {
        'model': name, 'family': m.family, 'size': m['size'], 'parameters': m.params, 'precision': 'float16',
        'path': m.path, 'source': m.source, 'prompt_set': PROMPT_SET, 'images': len(df),
        'model_load (s)': round(load_s, 1),
        'input_tokens (total)': int(df.input_tokens.sum()),
        'output_tokens (total)': int(df.output_tokens.sum()),
        'output_words (avg)': round(df.output_words.mean()),
        'responses_hit_token_limit': int(hit_limit),
        'time_to_first_token (ms, avg)': round(df['time_to_first_token (ms)'].mean(), 1),
        'prompt_processing_rate (tokens/s, avg)': round(df['prompt_processing_rate (tokens/s)'].mean(), 1),
        'inference_rate (tokens/s, avg)': round(df['inference_rate (tokens/s)'].mean(), 2),
        'inference_time (s, total)': round(df['inference_time (s)'].sum(), 2),
        'peak_memory (GB)': round(peak_gb, 2),
        'gpu_utilization (%, avg)': round(hw['GPU Utilization (%)'].mean(), 1),
        'gpu_power (W, avg)': round(hw['gpuPower'].mean(), 1),
        'gpu_temperature (C, max)': round(hw['GPU_Temperature(C)'].max(), 1),
        'cpu_utilization (%, avg)': round(hw['CPU_Util(%)'].mean(), 1),
        'cpu_power (W, avg)': round(hw['cpuPower'].mean(), 1),
        'npu_power (W, avg)': round(hw['npuPower'].mean(), 2),
        'system_energy (J)': round(hw['total_system_power'].sum() * 0.1, 1),   # 100 ms samples
    }
    (out_dir / f'{name}_simple_metrics.json').write_text(json.dumps(summary, indent=2))
    return summary, df

summary, per_prompt = run_model(MODEL)
display(per_prompt.drop(columns=['image', 'prompt', 'response']))
pd.Series(summary, name=MODEL)

## 5. Compare the family

Runs every other model of `FAMILY` that fits in GPU memory and shows the summaries side by side. Each model runs this
notebook in its own fresh process, so every model starts from the same clean GPU state. Set `ITT_COMPARE=0`
(or `COMPARE_FAMILY = False` in section 1) to skip.

In [ ]:
results = {MODEL: summary}
if COMPARE_FAMILY:
    for name, m in MODELS[MODELS.family == FAMILY].iterrows():
        if name == MODEL:
            continue
        if not m.fits:
            print(f'{name}: skipped, needs {m.fp16_gb} GB in float16 (GPU budget {GPU_BUDGET_GB:.1f} GB)')
            continue
        print(f'{name}: running in a fresh process...')
        env = {**os.environ, 'ITT_FAMILY': m.family, 'ITT_SIZE': m['size'], 'ITT_PROMPTS': PROMPT_SET,
               'ITT_MAX_TOKENS': str(MAX_NEW_TOKENS), 'ITT_COMPARE': '0'}
        with tempfile.TemporaryDirectory() as tmp:
            run = subprocess.run([sys.executable, '-m', 'nbconvert', '--to', 'notebook', '--execute',
                                  'notebooks/itt_image_to_text.ipynb', '--ExecutePreprocessor.timeout=-1',
                                  '--output-dir', tmp], env=env, capture_output=True, text=True)
        if run.returncode != 0:
            print(f'  failed:\n{run.stderr[-2000:]}')
            continue
        results[name] = json.loads((Path('outputs/itt') / m.family / m.params / PROMPT_SET /
                                    f'{name}_simple_metrics.json').read_text())
        print(f"  {results[name]['inference_rate (tokens/s, avg)']} tok/s, {results[name]['inference_time (s, total)']} s")
pd.DataFrame({name: results[name] for name in MODELS.index if name in results})

## 6. All models run so far

Combines every float16 model run on `PROMPT_SET` (any family) into `outputs/itt/all_models_<prompt set>.csv` and their
per-image metrics into `metrics_analysis/itt/all_models_prompt_metrics_<prompt set>.csv`.

In [ ]:
summaries = [json.loads(f.read_text()) for f in Path('outputs/itt').glob(f'*/*/{PROMPT_SET}/*_simple_metrics.json')]
summaries = sorted((s for s in summaries if s.get('precision') == 'float16'), key=lambda s: list(MODELS.index).index(s['model']))
all_models = pd.DataFrame(summaries)
all_models.to_csv(Path('outputs/itt') / f'all_models_{PROMPT_SET}.csv', index=False)
all_prompts = pd.concat([
    pd.read_csv(Path('metrics_analysis/itt') / s['family'] / s['parameters'] / PROMPT_SET / 'Model_Level_Metrics' /
                f"{s['model']}_prompt_metrics.csv", dtype={'prompt_id': str})
    for s in summaries])
all_prompts.to_csv(Path('metrics_analysis/itt') / f'all_models_prompt_metrics_{PROMPT_SET}.csv', index=False)
print(f'{len(all_models)} models, {len(all_prompts)} image rows')
all_models